In [30]:
# 加载环境变量
# 加载 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env")

True

# 定义模型
多模态模型

In [31]:
from langchain.chat_models import init_chat_model
import os

from langgraph.channels import topic

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url=os.environ.get("DASHSCOPE_API_URL")
)

# 定义工具

In [32]:

from langchain_tavily import TavilySearch
web_search = TavilySearch(
    max_results=5,
    topic="general",
    tavily_api_key=os.environ.get("TAVILY_API_KEY")
)


# 添加记忆管理


In [33]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 链接sqlite
connection = sqlite3.connect("resources/memory.db", check_same_thread=False)
# 初始化记忆管理
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()

# 定义智能体

In [34]:
from langchain.agents import create_agent

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份 “当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以 “可用食材清单” 为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。
请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

# 测试

In [35]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage([
    {"type": "text", "text":"帮我看看能做什么"},
    {"type": "image", "url": "https://ts3.tc.mm.bing.net/th/id/OIP-C.cwZmXC_2AeuJMpb8TVQ-ugHaLH?r=0&rs=1&pid=ImgDetMain&o=7&rm=3"},
])

config = {"configurable": {"thread_id": "2"}}

In [36]:
response = agent.invoke({"messages": [multimodal_message]}, config)

In [37]:
# 友好打印
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://ts3.tc.mm.bing.net/th/id/OIP-C.cwZmXC_2AeuJMpb8TVQ-ugHaLH?r=0&rs=1&pid=ImgDetMain&o=7&rm=3'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://ts3.tc.mm.bing.net/th/id/OIP-C.cwZmXC_2AeuJMpb8TVQ-ugHaLH?r=0&rs=1&pid=ImgDetMain&o=7&rm=3'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://ts3.tc.mm.bing.net/th/id/OIP-C.cwZmXC_2AeuJMpb8TVQ-ugHaLH?r=0&rs=1&pid=ImgDetMain&o=7&rm=3'}]
================================== Ai Message ==================================

## 🧊 冰箱食材识别

根据照片，我为您整理出以下**当前可用食材清单**：

| 类别 | 食材 |
|------|------|
| 🥬 蔬菜 | 红椒、黄瓜、番茄、胡萝卜、芹菜、红辣椒、西兰花、黄豆芽、小番茄 |
| 🥚 蛋类 | 鸡蛋（一整盒） |
| 🍗 肉类 | 整鸡、香肠/腊肠、牛肉块、猪肉块、切好的肉丁 |
| 🍎 水果

In [38]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第一道菜，给我详细介绍一下")]},
    config
)

In [39]:
# 友好打印
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://ts3.tc.mm.bing.net/th/id/OIP-C.cwZmXC_2AeuJMpb8TVQ-ugHaLH?r=0&rs=1&pid=ImgDetMain&o=7&rm=3'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://ts3.tc.mm.bing.net/th/id/OIP-C.cwZmXC_2AeuJMpb8TVQ-ugHaLH?r=0&rs=1&pid=ImgDetMain&o=7&rm=3'}]
================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么'}, {'type': 'image', 'url': 'https://ts3.tc.mm.bing.net/th/id/OIP-C.cwZmXC_2AeuJMpb8TVQ-ugHaLH?r=0&rs=1&pid=ImgDetMain&o=7&rm=3'}]
================================== Ai Message ==================================

## 🧊 冰箱食材识别

根据照片，我为您整理出以下**当前可用食材清单**：

| 类别 | 食材 |
|------|------|
| 🥬 蔬菜 | 红椒、黄瓜、番茄、胡萝卜、芹菜、红辣椒、西兰花、黄豆芽、小番茄 |
| 🥚 蛋类 | 鸡蛋（一整盒） |
| 🍗 肉类 | 整鸡、香肠/腊肠、牛肉块、猪肉块、切好的肉丁 |
| 🍎 水果

In [40]:
# response['message'][-1].pretty_print()

KeyError: 'message'